In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## CVM - Fundos Imobiliarios - Complemento

In [0]:
bronze_path_complemento = "/Volumes/workspace/case_spark_cvm/bronze/cvm_fii_complemento/"

df_silver_fii_complemento = ler_ultima_particao_delta(spark, bronze_path_complemento)

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_fii_complemento = df_silver_fii_complemento.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_silver_fii_complemento = df_silver_fii_complemento.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['Data_Referencia', 'CNPJ_Fundo_Classe']

# Aplicando a filtro para dropar as colunas
df_silver_fii_complemento = df_silver_fii_complemento.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
# Dropando a Data de Processamento da Bronze 
df_silver_fii_complemento = df_silver_fii_complemento.drop("data_processamento")

# Criando a Data de Processamento da silver
df_silver_fii_complemento = df_silver_fii_complemento.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

In [0]:
df_silver_fii_complemento = df_silver_fii_complemento\
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType())) \
    .withColumn('data_referencia', f.col('Data_Referencia').cast(t.DateType())) \
    .withColumn('versao', f.col('Versao').cast(t.IntegerType())) \
    .withColumn('data_informacao_numero_cotistas', f.col('Data_Informacao_Numero_Cotistas').cast(t.DateType())) \
    .withColumn('total_numero_cotistas', f.col('Total_Numero_Cotistas').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_pessoa_fisica', f.col('Numero_Cotistas_Pessoa_Fisica').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_pessoa_juridica_nao_financeira', f.col('Numero_Cotistas_Pessoa_Juridica_Nao_Financeira').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_banco_comercial', f.col('Numero_Cotistas_Banco_Comercial').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_corretora_distribuidora', f.col('Numero_Cotistas_Corretora_Distribuidora').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outras_pessoas_juridicas_financeira', f.col('Numero_Cotistas_Outras_Pessoas_Juridicas_Financeira').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_investidores_nao_residentes', f.col('Numero_Cotistas_Investidores_Nao_Residentes').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_entidade_aberta_previdencia_complementar', f.col('Numero_Cotistas_Entidade_Aberta_Previdencia_Complementar').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_entidade_fechada_previdencia_complementar', f.col('Numero_Cotistas_Entidade_Fechada_Previd�ncia_Complementar').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_regime_proprio_previdencia_servidores_publicos', f.col('Numero_Cotistas_Regime_Proprio_Previdencia_Servidores_Publicos').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_sociedade_seguradora_resseguradora', f.col('Numero_Cotistas_Sociedade_Seguradora_Resseguradora').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_sociedade_capitalizacao_arrendamento_mercantil', f.col('Numero_Cotistas_Sociedade_Capitalizacao_Arrendamento_Mercantil').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_fii', f.col('Numero_Cotistas_FII').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outros_fundos', f.col('Numero_Cotistas_Outros_Fundos').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_distribuidores_fundo', f.col('Numero_Cotistas_Distribuidores_Fundo').cast(t.IntegerType())) \
    .withColumn('numero_cotistas_outros_tipos', f.col('Numero_Cotistas_Outros_Tipos').cast(t.IntegerType())) \
    .withColumn('valor_ativo', f.col('Valor_Ativo').cast(t.DecimalType(22, 2))) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(22, 2))) \
    .withColumn('cotas_emitidas', f.col('Cotas_Emitidas').cast(t.DecimalType(22, 2))) \
    .withColumn('valor_patrimonial_cotas', f.col('Valor_Patrimonial_Cotas').cast(t.DecimalType(22, 2))) \
    .withColumn('percentual_despesas_taxa_administracao', f.col('Percentual_Despesas_Taxa_Administracao').cast("double")) \
    .withColumn('percentual_despesas_agente_custodiante', f.col('Percentual_Despesas_Agente_Custodiante').cast("double")) \
    .withColumn('percentual_rentabilidade_efetiva_mes', f.col('Percentual_Rentabilidade_Efetiva_Mes').cast("double")) \
    .withColumn('percentual_rentabilidade_patrimonial_mes', f.col('Percentual_Rentabilidade_Patrimonial_Mes').cast("double")) \
    .withColumn('percentual_dividend_yield_mes', f.col('Percentual_Dividend_Yield_Mes').cast("double")) \
    .withColumn('percentual_amortizacao_cotas_mes', f.col('Percentual_Amortizacao_Cotas_Mes').cast("double")) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_silver_fii_complemento.write \
    .mode('overwrite')\
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fii_complemento")